# Data Stats

Computes the summary statistics described in `data/README.md`.

In [1]:
import json
from pathlib import Path
from collections import Counter, defaultdict

DATA_DIR = Path(".")  # run from data/
TRANSCRIPTS = DATA_DIR / "transcripts" / "step_up.jsonl"
ANNOTATIONS = DATA_DIR / "teacher_annotations" / "step_up_annotations.jsonl"

## transcripts/step_up.jsonl

In [2]:
transcripts = []
with open(TRANSCRIPTS) as f:
    for line in f:
        line = line.strip()
        if line:
            transcripts.append(json.loads(line))

print(f"Total records: {len(transcripts):,}")

Total records: 21,445


In [3]:
# Records per batch
batch_counts = Counter(r["batch"] for r in transcripts)
print("Records per batch:")
for batch in sorted(batch_counts):
    print(f"  {batch}: {batch_counts[batch]:,}")

Records per batch:
  2025-10-16: 462
  2026-02-13: 109
  2026-03-24: 10,878
  2026-04-27: 9,996


In [4]:
# Turns per session
turn_counts = [len(r["turns"]) for r in transcripts]
avg_turns = sum(turn_counts) / len(turn_counts)
print(f"Turns per session — avg: {avg_turns:.0f}, min: {min(turn_counts)}, max: {max(turn_counts)}")

Turns per session — avg: 349, min: 5, max: 1789


In [5]:
# Whitespace-tokenized token length per turn
all_turn_lengths = []
for r in transcripts:
    for t in r["turns"]:
        text = t.get("text", "") or ""
        all_turn_lengths.append(len(text.split()))

avg_tok = sum(all_turn_lengths) / len(all_turn_lengths)
print(f"Whitespace-token length per turn — avg: {avg_tok:.1f}, min: {min(all_turn_lengths)}, max: {max(all_turn_lengths)}")
print(f"(across {len(all_turn_lengths):,} turns total)")

Whitespace-token length per turn — avg: 10.8, min: 0, max: 3256
(across 7,492,218 turns total)


In [6]:
# has_video distribution
video_counts = Counter(r["has_video"] for r in transcripts)
print(f"has_video=True:  {video_counts[True]:,} ({100*video_counts[True]/len(transcripts):.1f}%)")
print(f"has_video=False: {video_counts[False]:,} ({100*video_counts[False]/len(transcripts):.1f}%)")

has_video=True:  20,983 (97.8%)
has_video=False: 462 (2.2%)


In [7]:
# Session duration distribution (in minutes)
durations = []
for r in transcripts:
    turns = r["turns"]
    if turns:
        duration_sec = turns[-1]["end_seconds"] - turns[0]["start_seconds"]
        durations.append(duration_sec / 60)

avg_dur = sum(durations) / len(durations)
print(f"Session duration (minutes) — avg: {avg_dur:.1f}, min: {min(durations):.1f}, max: {max(durations):.1f}")

Session duration (minutes) — avg: 52.8, min: 0.9, max: 180.4


In [8]:
# Unique tutors and students
tutor_ids = {r["session"]["tutor_id"] for r in transcripts}
student_ids = {r["session"]["student_id"] for r in transcripts}
print(f"Unique tutors:  {len(tutor_ids):,}")
print(f"Unique students: {len(student_ids):,}")

Unique tutors:  1,116
Unique students: 1,302


## teacher_annotations/step_up_annotations.jsonl

In [9]:
annotations = []
with open(ANNOTATIONS) as f:
    for line in f:
        line = line.strip()
        if line:
            annotations.append(json.loads(line))

print(f"Total records: {len(annotations):,}")

Total records: 1,354


In [10]:
# Records per annotation type
type_counts = Counter(r["annotation_type"] for r in annotations)
print("Records per annotation_type:")
for atype in sorted(type_counts):
    print(f"  {atype}: {type_counts[atype]}")

Records per annotation_type:
  caption: 79
  rapport: 801
  scaffolding: 474


In [11]:
# Records per batch
ann_batch_counts = Counter(r["batch"] for r in annotations)
print("Records per batch:")
for batch in sorted(ann_batch_counts):
    print(f"  {batch}: {ann_batch_counts[batch]}")

Records per batch:
  2025-10-16: 1354


In [12]:
# Null turn_number_start/end in turn_annotations, by type
for atype in ("rapport", "scaffolding"):
    records = [r for r in annotations if r["annotation_type"] == atype]
    total_ta = sum(len(r["turn_annotations"]) for r in records)
    null_ta = sum(
        1
        for r in records
        for ta in r["turn_annotations"]
        if ta.get("turn_number_start") is None and ta.get("turn_number_end") is None
    )
    pct = 100 * null_ta / total_ta if total_ta else 0
    print(f"{atype}: {null_ta} / {total_ta} null-anchored turn_annotations ({pct:.1f}%)")

rapport: 27 / 5653 null-anchored turn_annotations (0.5%)
scaffolding: 15 / 3121 null-anchored turn_annotations (0.5%)


In [13]:
# Cut point coverage by annotation type
for atype in ("scaffolding", "rapport", "caption"):
    records = [r for r in annotations if r["annotation_type"] == atype]
    total_ta = sum(len(r["turn_annotations"]) for r in records)
    records_with_cut = sum(
        1 for r in records
        if any("cut_turn" in ta for ta in r["turn_annotations"])
    )
    turns_with_cut = sum(
        1
        for r in records
        for ta in r["turn_annotations"]
        if "cut_turn" in ta
    )
    print(f"{atype}: {records_with_cut} / {len(records)} records have cut_turn, "
          f"{turns_with_cut} / {total_ta} turn_annotation items")

scaffolding: 295 / 474 records have cut_turn, 2000 / 3121 turn_annotation items
rapport: 692 / 801 records have cut_turn, 5034 / 5653 turn_annotation items
caption: 0 / 79 records have cut_turn, 0 / 5547 turn_annotation items


In [14]:
# SAR field whitespace-token lengths (scaffolding + rapport only)
sar_fields = {"situation": [], "action": [], "result": []}
for r in annotations:
    if r["annotation_type"] not in ("scaffolding", "rapport"):
        continue
    for ta in r["turn_annotations"]:
        for field in sar_fields:
            text = ta.get(field) or ""
            sar_fields[field].append(len(text.split()))

print("SAR field whitespace-token lengths (scaffolding + rapport, all turn_annotations):")
for field, lengths in sar_fields.items():
    avg = sum(lengths) / len(lengths)
    print(f"  {field}: avg {avg:.1f}, min {min(lengths)}, max {max(lengths)}")

SAR field whitespace-token lengths (scaffolding + rapport, all turn_annotations):
  situation: avg 24.0, min 1, max 256
  action: avg 19.4, min 1, max 342
  result: avg 30.1, min 1, max 308


In [15]:
# moment_id <-> (conv_id, turn_start, turn_end) mapping
moment_to_coords = defaultdict(set)
coords_to_moments = defaultdict(set)

for r in annotations:
    if r.get("annotation_type") not in ("scaffolding", "rapport"):
        continue
    conv_id = r["transcript_id"]
    for ta in r.get("turn_annotations", []):
        if "moment_id" not in ta:
            continue
        mid = ta["moment_id"]
        coords = (conv_id, ta.get("turn_number_start"), ta.get("turn_number_end"))
        moment_to_coords[mid].add(coords)
        coords_to_moments[coords].add(mid)

multi_coord = {mid: c for mid, c in moment_to_coords.items() if len(c) > 1}
multi_moment = {c: m for c, m in coords_to_moments.items() if len(m) > 1}

print(f"Unique moment_ids:                    {len(moment_to_coords)}")
print(f"Unique (conv, turn_start, turn_end):  {len(coords_to_moments)}")
print(f"moment_ids mapping to >1 coord:       {len(multi_coord)}  (moment_id -> coords is injective)")
print(f"coords mapping to >1 moment_id:       {len(multi_moment)}  (coords -> moment_id is NOT injective)")
print()
print("Explanation: two annotators can independently annotate the same turn span,")
print("producing distinct moment_ids for the same (conv, turn_start, turn_end).")

Unique moment_ids:                    1747
Unique (conv, turn_start, turn_end):  1714
moment_ids mapping to >1 coord:       0  (moment_id -> coords is injective)
coords mapping to >1 moment_id:       30  (coords -> moment_id is NOT injective)

Explanation: two annotators can independently annotate the same turn span,
producing distinct moment_ids for the same (conv, turn_start, turn_end).


In [16]:
# Annotated sessions (unique transcript_ids with scaffolding or rapport)
annotated_ids = {r["transcript_id"] for r in annotations if r["annotation_type"] in ("scaffolding", "rapport")}
print(f"Sessions with scaffolding or rapport annotations: {len(annotated_ids)}")

# Moments per type (excluding null-anchored)
for atype in ("scaffolding", "rapport"):
    records = [r for r in annotations if r["annotation_type"] == atype]
    moments = [
        ta
        for r in records
        for ta in r["turn_annotations"]
        if ta.get("turn_number_start") is not None
    ]
    print(f"{atype} moments (anchored): {len(moments)}")

Sessions with scaffolding or rapport annotations: 209
scaffolding moments (anchored): 3106
rapport moments (anchored): 5626


## Train/Test Split Statistics

In [17]:
import sys
sys.path.insert(0, str(Path(".").resolve().parent))  # repo root
from annotator.core.utils import compute_iou

SPLIT_FILE = DATA_DIR / "split.json"
GT_DIR = DATA_DIR / "ground_truth_hybrid"

with open(SPLIT_FILE) as f:
    split = json.load(f)

def load_split_moments(conv_ids):
    moments = []
    for conv_id in conv_ids:
        path = GT_DIR / f"{conv_id}.json"
        if not path.exists():
            continue
        with open(path) as f:
            data = json.load(f)
        for m in data.get("key_moments", []):
            moments.append({**m, "conversation_id": conv_id})
    return moments

train_moments = load_split_moments(split["train"])
test_moments = load_split_moments(split["test"])
print(f"Train: {len(split['train'])} conversations, {len(train_moments)} moments")
print(f"Test:  {len(split['test'])} conversations, {len(test_moments)} moments")

Train: 102 conversations, 4615 moments
Test:  105 conversations, 4109 moments


In [18]:
def split_stats(moments, conv_ids, label):
    total = len(moments)
    print(f"{'='*50}")
    print(f"{label}  ({len(conv_ids)} conversations, {total} moments)")
    print(f"{'='*50}")

    # Annotation type breakdown with per-type label distribution
    type_counts = Counter(m["annotation_type"] for m in moments)
    print("\nAnnotation type and strategy label distribution:")
    for atype in ("scaffolding", "rapport"):
        n = type_counts[atype]
        tc = Counter(m["strategy_label"] for m in moments if m["annotation_type"] == atype)
        print(f"  {atype}: {n}")
        for lbl in ("effective", "partial", "ineffective"):
            print(f"    {lbl}: {tc[lbl]} ({100*tc[lbl]/n:.1f}%)")

    # Unique key moment spans (by conv_id, turn_start, turn_end)
    print("\nUnique key moment spans (by conv_id, turn_start, turn_end):")
    for atype in ("scaffolding", "rapport"):
        type_moments = [m for m in moments if m["annotation_type"] == atype]
        unique = len({(m["conversation_id"], m["turn_start"], m["turn_end"]) for m in type_moments})
        print(f"  {atype}: {unique}")

    # Count moments with exact turn_start/turn_end match from a different annotator
    # within the same conversation and annotation type.
    by_conv = defaultdict(list)
    for i, m in enumerate(moments):
        by_conv[m["conversation_id"]].append((i, m))

    exact_match_indices = set()
    for conv_moments in by_conv.values():
        for idx_a, (gi_a, m1) in enumerate(conv_moments):
            for gi_b, m2 in conv_moments[idx_a + 1:]:
                if m1["annotator_id"] == m2["annotator_id"]:
                    continue
                if m1["annotation_type"] != m2["annotation_type"]:
                    continue
                if m1["turn_start"] == m2["turn_start"] and m1["turn_end"] == m2["turn_end"]:
                    exact_match_indices.add(gi_a)
                    exact_match_indices.add(gi_b)

    print("\nMoments with matching start/end across >1 annotator:")
    for atype in ("scaffolding", "rapport"):
        type_indices = {i for i, m in enumerate(moments) if m["annotation_type"] == atype}
        n_total = len(type_indices)
        n_match = len(exact_match_indices & type_indices)
        print(f"  {atype}: {n_match} ({100*n_match/n_total:.1f}%)")

split_stats(train_moments, split["train"], "TRAIN")
print()
split_stats(test_moments, split["test"], "TEST")

TRAIN  (102 conversations, 4615 moments)

Annotation type and strategy label distribution:
  scaffolding: 1628
    effective: 653 (40.1%)
    partial: 299 (18.4%)
    ineffective: 671 (41.2%)
  rapport: 2987
    effective: 1451 (48.6%)
    partial: 693 (23.2%)
    ineffective: 789 (26.4%)

Unique key moment spans (by conv_id, turn_start, turn_end):
  scaffolding: 866
  rapport: 468

Moments with matching start/end across >1 annotator:
  scaffolding: 1207 (74.1%)
  rapport: 2987 (100.0%)

TEST  (105 conversations, 4109 moments)

Annotation type and strategy label distribution:
  scaffolding: 1478
    effective: 575 (38.9%)
    partial: 278 (18.8%)
    ineffective: 606 (41.0%)
  rapport: 2631
    effective: 1227 (46.6%)
    partial: 627 (23.8%)
    ineffective: 749 (28.5%)

Unique key moment spans (by conv_id, turn_start, turn_end):
  scaffolding: 670
  rapport: 459

Moments with matching start/end across >1 annotator:
  scaffolding: 1154 (78.1%)
  rapport: 2631 (100.0%)


## Decomposed Facets (Scaffolding only)

In [19]:
def decomposed_facet_stats(moments, label):
    total = len(moments)
    action_counts = [len(m.get("action_decomposed") or []) for m in moments]
    result_counts = [len(m.get("result_decomposed") or []) for m in moments]
    n_with_action = sum(1 for c in action_counts if c > 0)
    n_with_result = sum(1 for c in result_counts if c > 0)
    print(f"{label} ({total} scaffolding moments)")
    print(f"  Tutor actions (action_decomposed):")
    print(f"    moments with >=1 facet: {n_with_action} ({100*n_with_action/total:.1f}%)")
    print(f"    avg facets/moment:      {sum(action_counts)/total:.2f}  (max: {max(action_counts)})")
    print(f"  Student indicators (result_decomposed):")
    print(f"    moments with >=1 facet: {n_with_result} ({100*n_with_result/total:.1f}%)")
    print(f"    avg facets/moment:      {sum(result_counts)/total:.2f}  (max: {max(result_counts)})")

def load_scaffolding_moments(conv_ids):
    moments = []
    for conv_id in conv_ids:
        path = GT_DIR / f"{conv_id}.json"
        if not path.exists():
            continue
        with open(path) as f:
            data = json.load(f)
        for m in data.get("key_moments", []):
            if m["annotation_type"] == "scaffolding":
                moments.append(m)
    return moments

all_scaffolding = load_scaffolding_moments(
    [p.stem for p in GT_DIR.glob("*.json")]
)
decomposed_facet_stats(all_scaffolding, "ALL")
print()
decomposed_facet_stats(load_scaffolding_moments(split["train"]), "TRAIN")
print()
decomposed_facet_stats(load_scaffolding_moments(split["test"]), "TEST")

ALL (3106 scaffolding moments)
  Tutor actions (action_decomposed):
    moments with >=1 facet: 2741 (88.2%)
    avg facets/moment:      1.72  (max: 12)
  Student indicators (result_decomposed):
    moments with >=1 facet: 1718 (55.3%)
    avg facets/moment:      0.83  (max: 6)

TRAIN (1628 scaffolding moments)
  Tutor actions (action_decomposed):
    moments with >=1 facet: 1465 (90.0%)
    avg facets/moment:      1.77  (max: 12)
  Student indicators (result_decomposed):
    moments with >=1 facet: 936 (57.5%)
    avg facets/moment:      0.88  (max: 6)

TEST (1478 scaffolding moments)
  Tutor actions (action_decomposed):
    moments with >=1 facet: 1276 (86.3%)
    avg facets/moment:      1.66  (max: 9)
  Student indicators (result_decomposed):
    moments with >=1 facet: 782 (52.9%)
    avg facets/moment:      0.78  (max: 6)
